In [ ]:
# --- Cell 1/5: clone + install ---------------------------------------------
import sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/swaroopms658/loralink-reviewer.git"   # public
if IN_COLAB:
    get_ipython().system(f"git clone --depth 1 {REPO_URL} loralink")
    get_ipython().run_line_magic("cd", "loralink")
    get_ipython().system("pip -q install -r loralink_reviewer_response/requirements-colab.txt")
    sys.path.insert(0, ".")
else:
    print("local run: expecting to be launched from the repo root with results/ pre-populated")
    sys.path.insert(0, ".")

In [ ]:
# --- Cell 2/5: upload the downloaded results_*.csv into results/ ----------
# Upload every results_* file. aggregate.py reads results_stat_/quality_/sched_/
# scale_/net_/converge_; results_qsys_* is retained as raw Phi-1.5 per-batch
# training-log evidence (loss/latency/compression) for the appendix - it is not
# aggregated into any table.
import glob, os, shutil
os.makedirs("results", exist_ok=True)
if IN_COLAB:
    from google.colab import files
    up = files.upload()
    landed = []
    for name in up:
        base = os.path.basename(name)
        if base.startswith("results_"):
            shutil.move(name, os.path.join("results", base))
            landed.append(base)
        else:
            print(f"skipped {base} (name does not start with 'results_')")
    print(f"landed {len(landed)} file(s) in results/: {sorted(landed)}")
else:
    print("local run: using pre-populated results/ ->", sorted(glob.glob("results/results_*")))

In [ ]:
# --- Cell 3/5: aggregate every shard into figures/ + summary.json --------
from loralink_reviewer_response.aggregate import build_all, render_response

s = build_all("results",
              "loralink_reviewer_response/baselines/published_baselines.csv",
              "figures")
print("summary keys:", sorted(s))

In [ ]:
# --- Cell 4/5: fill RESPONSE_ABHAY_NIKHIL.md from summary.json -----------
# WARN: unfilled placeholder(s): ... below means a shard did not come back.
render_response("figures/summary.json",
                "loralink_reviewer_response/RESPONSE_ABHAY_NIKHIL.md",
                "RESPONSE_ABHAY_NIKHIL.filled.md")
print(open("RESPONSE_ABHAY_NIKHIL.filled.md", encoding="utf-8").read())

In [ ]:
# --- Cell 5/5: show every figure inline, then download everything -------
import glob
from IPython.display import Image, display

for png in sorted(glob.glob("figures/*.png")):
    print(png)
    display(Image(filename=png))

if IN_COLAB:
    from google.colab import files
    for f in sorted(glob.glob("figures/*")) + ["RESPONSE_ABHAY_NIKHIL.filled.md"]:
        files.download(f)
else:
    print("local run: skipping download; outputs are in figures/ and RESPONSE_ABHAY_NIKHIL.filled.md")